In [1]:
# ============================================================
# 11_AURORA_statistical_significance_and_block_bootstrap.ipynb
# AURORA-TWETF Statistical Significance and Block Bootstrap
#
# Purpose:
# 1. Load Notebook 10 UAMV-vs-benchmark return outputs.
# 2. Test whether AURORA10 risk-adjusted improvements are statistically meaningful.
# 3. Use paired block bootstrap for dependent financial returns.
# 4. Estimate confidence intervals for:
#    - total return difference
#    - annual return difference
#    - Sharpe difference
#    - Sortino difference
#    - max drawdown difference
#    - Calmar difference
#    - mean daily excess return
# 5. Produce HAC/Newey-West style paired mean-return tests.
# 6. Save paper-ready tables, figures, diagnostics, validation report,
#    and SHA-256 manifest.
#
# Important:
# - Educational/research backtest only.
# - Not personalized financial advice.
# - Statistical tests are exploratory because the test period is short.
# - Block bootstrap preserves some serial dependence but is not a guarantee
#   of true future outperformance.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy import stats
    HAS_SCIPY_STATS = True
except Exception:
    HAS_SCIPY_STATS = False
    print("scipy.stats not available. Some p-values will be skipped.")

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK08_RUN_ID = "20260624_034204"
NOTEBOOK08B_RUN_ID = "20260624_070827"
NOTEBOOK09_RUN_ID = "20260624_072914"
NOTEBOOK10_RUN_ID = "20260624_100748"

NOTEBOOK10_ROOT = OUTPUT_ROOT / "uncertainty_aware_mean_variance_allocation" / f"run_{NOTEBOOK10_RUN_ID}"
NOTEBOOK10_RETURN_DIR = NOTEBOOK10_ROOT / "returns"
NOTEBOOK10_TABLE_DIR = NOTEBOOK10_ROOT / "tables"

NOTEBOOK10_COMPARISON_RETURNS = NOTEBOOK10_RETURN_DIR / "comparison_returns_uamv_vs_benchmarks.parquet"
NOTEBOOK10_COMPARISON_RANKINGS = TABLE_DIR / f"table_93_comparison_rankings_uamv_vs_benchmarks_{NOTEBOOK10_RUN_ID}.csv"
NOTEBOOK10_AURORA_VS_BENCHMARK = TABLE_DIR / f"table_94_uamv_aurora_vs_best_benchmark_{NOTEBOOK10_RUN_ID}.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "statistical_significance_block_bootstrap" / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
PLOT_DIR = RUN_ROOT / "plots"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"
BOOTSTRAP_DIR = RUN_ROOT / "bootstrap_samples"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    PLOT_DIR,
    PAPER_FIGURE_DIR,
    DIAGNOSTIC_DIR,
    BOOTSTRAP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 11: Statistical Significance and Block Bootstrap")
print("=" * 80)
print("Timestamp UTC              :", RUN_TIMESTAMP)
print("Run ID                     :", RUN_ID)
print("Notebook 10 root           :", NOTEBOOK10_ROOT)
print("Notebook 10 comparison ret :", NOTEBOOK10_COMPARISON_RETURNS)
print("Notebook 10 rankings       :", NOTEBOOK10_COMPARISON_RANKINGS)
print("Run root                   :", RUN_ROOT)
print("=" * 80)

for required_path in [
    NOTEBOOK10_ROOT,
    NOTEBOOK10_COMPARISON_RETURNS,
    NOTEBOOK10_COMPARISON_RANKINGS,
]:
    if not Path(required_path).exists():
        raise FileNotFoundError(f"Required path not found: {required_path}")

# ============================================================
# 2. Statistical configuration
# ============================================================

ANNUALIZATION_DAYS = 252

PRIMARY_AURORA_POLICY = "AURORA10_UAMV_B_more60_defensive"

PRIMARY_BENCHMARKS = [
    "B6_00881_only",
    "B3_0050_only",
    "B1_equal_weight_all_etfs",
    "B10_momentum_top2_63d",
    "B16_minimum_variance_126d",
]

ADDITIONAL_AURORA_POLICIES = [
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    "AURORA10_UAMV_A_balanced",
    "AURORA10_UAMV_C_more60_growth",
    "AURORA10_validation_selected_UAMV",
]

N_BOOTSTRAP = 5000
BLOCK_LENGTHS = [5, 10, 20, 40]
PRIMARY_BLOCK_LENGTH = 20
RANDOM_STATE = 42

CONFIDENCE_LEVELS = [0.90, 0.95]
EPS = 1e-12

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

def read_table_auto(path):
    path = Path(path)

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    df.index.name = "date"
    return df.sort_index()

def annualized_return_from_daily(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return np.nan

    total = float((1.0 + r).prod() - 1.0)
    return float((1.0 + total) ** (ANNUALIZATION_DAYS / n) - 1.0)

def annualized_volatility(r):
    r = pd.Series(r).dropna().astype(float)
    if len(r) <= 1:
        return np.nan
    return float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS))

def sharpe_ratio(r):
    ar = annualized_return_from_daily(r)
    av = annualized_volatility(r)

    if not np.isfinite(av) or av <= 0:
        return np.nan

    return float(ar / av)

def sortino_ratio(r):
    r = pd.Series(r).dropna().astype(float)
    ar = annualized_return_from_daily(r)
    downside = r[r < 0]

    if len(downside) <= 1:
        return np.nan

    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS))

    if not np.isfinite(downside_vol) or downside_vol <= 0:
        return np.nan

    return float(ar / downside_vol)

def equity_curve(r):
    r = pd.Series(r).dropna().astype(float)
    return (1.0 + r).cumprod()

def max_drawdown(r):
    eq = equity_curve(r)

    if len(eq) == 0:
        return np.nan

    dd = eq / eq.cummax() - 1.0
    return float(dd.min())

def calmar_ratio(r):
    ar = annualized_return_from_daily(r)
    mdd = max_drawdown(r)

    if not np.isfinite(mdd) or mdd >= 0:
        return np.nan

    return float(ar / abs(mdd))

def total_return(r):
    r = pd.Series(r).dropna().astype(float)
    if len(r) == 0:
        return np.nan
    return float((1.0 + r).prod() - 1.0)

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)

    if len(r) == 0:
        return {}

    return {
        "n_days": int(len(r)),
        "total_return": total_return(r),
        "annual_return": annualized_return_from_daily(r),
        "annual_volatility": annualized_volatility(r),
        "sharpe_ratio": sharpe_ratio(r),
        "sortino_ratio": sortino_ratio(r),
        "max_drawdown": max_drawdown(r),
        "calmar_ratio": calmar_ratio(r),
        "mean_daily_return": float(r.mean()),
        "std_daily_return": float(r.std(ddof=1)) if len(r) > 1 else np.nan,
        "hit_rate": float((r > 0).mean()),
    }

def paired_difference_metrics(aurora_r, benchmark_r):
    aurora_r = pd.Series(aurora_r).astype(float)
    benchmark_r = pd.Series(benchmark_r).astype(float)

    common = aurora_r.index.intersection(benchmark_r.index)
    a = aurora_r.loc[common]
    b = benchmark_r.loc[common]
    d = a - b

    out = {
        "n_days": int(len(common)),
        "aurora_total_return": total_return(a),
        "benchmark_total_return": total_return(b),
        "diff_total_return": total_return(a) - total_return(b),
        "aurora_annual_return": annualized_return_from_daily(a),
        "benchmark_annual_return": annualized_return_from_daily(b),
        "diff_annual_return": annualized_return_from_daily(a) - annualized_return_from_daily(b),
        "aurora_annual_volatility": annualized_volatility(a),
        "benchmark_annual_volatility": annualized_volatility(b),
        "diff_annual_volatility": annualized_volatility(a) - annualized_volatility(b),
        "aurora_sharpe": sharpe_ratio(a),
        "benchmark_sharpe": sharpe_ratio(b),
        "diff_sharpe": sharpe_ratio(a) - sharpe_ratio(b),
        "aurora_sortino": sortino_ratio(a),
        "benchmark_sortino": sortino_ratio(b),
        "diff_sortino": sortino_ratio(a) - sortino_ratio(b),
        "aurora_max_drawdown": max_drawdown(a),
        "benchmark_max_drawdown": max_drawdown(b),
        "drawdown_improvement": max_drawdown(a) - max_drawdown(b),
        "aurora_calmar": calmar_ratio(a),
        "benchmark_calmar": calmar_ratio(b),
        "diff_calmar": calmar_ratio(a) - calmar_ratio(b),
        "mean_daily_excess_return": float(d.mean()),
        "annualized_mean_excess_return": float(d.mean() * ANNUALIZATION_DAYS),
        "excess_return_hit_rate": float((d > 0).mean()),
    }

    return out

def circular_block_indices(n, block_length, rng):
    if n <= 0:
        raise ValueError("n must be positive.")

    if block_length <= 0:
        raise ValueError("block_length must be positive.")

    idx = []

    while len(idx) < n:
        start = rng.integers(0, n)
        block = [(start + j) % n for j in range(block_length)]
        idx.extend(block)

    return np.asarray(idx[:n], dtype=int)

def iid_indices(n, rng):
    return rng.integers(0, n, size=n)

def bootstrap_p_value_greater_zero(samples):
    samples = np.asarray(samples, dtype=float)
    samples = samples[np.isfinite(samples)]

    if len(samples) == 0:
        return np.nan, np.nan

    p_le_zero = float(np.mean(samples <= 0.0))
    p_ge_zero = float(np.mean(samples >= 0.0))

    # One-sided p-value for H1: statistic > 0.
    p_one_sided_positive = p_le_zero

    # Two-sided approximate bootstrap p-value.
    p_two_sided = float(2.0 * min(p_le_zero, p_ge_zero))
    p_two_sided = min(1.0, p_two_sided)

    return p_one_sided_positive, p_two_sided

def confidence_interval(samples, level):
    samples = np.asarray(samples, dtype=float)
    samples = samples[np.isfinite(samples)]

    if len(samples) == 0:
        return np.nan, np.nan

    alpha = 1.0 - level
    lo = np.quantile(samples, alpha / 2.0)
    hi = np.quantile(samples, 1.0 - alpha / 2.0)

    return float(lo), float(hi)

def newey_west_se(x, max_lag=None):
    x = pd.Series(x).dropna().astype(float).values
    n = len(x)

    if n <= 1:
        return np.nan

    x_centered = x - x.mean()

    if max_lag is None:
        max_lag = int(np.floor(4 * (n / 100.0) ** (2.0 / 9.0)))
        max_lag = max(1, max_lag)

    gamma0 = np.sum(x_centered * x_centered) / n
    var = gamma0

    for lag in range(1, max_lag + 1):
        weight = 1.0 - lag / (max_lag + 1.0)
        gamma = np.sum(x_centered[lag:] * x_centered[:-lag]) / n
        var += 2.0 * weight * gamma

    se_mean = math.sqrt(max(var, 0.0) / n)
    return float(se_mean)

def newey_west_mean_test(x, max_lag=None):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)

    if n <= 2:
        return {
            "n": n,
            "mean": np.nan,
            "nw_se": np.nan,
            "t_stat": np.nan,
            "p_value_two_sided_normal": np.nan,
            "annualized_mean": np.nan,
            "max_lag": max_lag,
        }

    if max_lag is None:
        max_lag = int(np.floor(4 * (n / 100.0) ** (2.0 / 9.0)))
        max_lag = max(1, max_lag)

    mean_x = float(x.mean())
    se = newey_west_se(x, max_lag=max_lag)

    if not np.isfinite(se) or se <= 0:
        t_stat = np.nan
        p_val = np.nan
    else:
        t_stat = mean_x / se
        p_val = float(2.0 * (1.0 - stats.norm.cdf(abs(t_stat)))) if HAS_SCIPY_STATS else np.nan

    return {
        "n": int(n),
        "mean": mean_x,
        "nw_se": se,
        "t_stat": float(t_stat) if np.isfinite(t_stat) else np.nan,
        "p_value_two_sided_normal": p_val,
        "annualized_mean": mean_x * ANNUALIZATION_DAYS,
        "max_lag": int(max_lag),
    }

# ============================================================
# 4. Load Notebook 10 results
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading Notebook 10 comparison returns and rankings")
print("=" * 80)

comparison_returns_raw = read_table_auto(NOTEBOOK10_COMPARISON_RETURNS)
comparison_rank_df = pd.read_csv(NOTEBOOK10_COMPARISON_RANKINGS)

if "policy_name" not in comparison_returns_raw.columns:
    raise ValueError("comparison_returns file must include policy_name column.")

required_return_cols = ["policy_name", "net_return", "equity", "drawdown"]
missing_cols = [c for c in required_return_cols if c not in comparison_returns_raw.columns]
if missing_cols:
    raise ValueError(f"comparison_returns missing columns: {missing_cols}")

print("Comparison returns shape:", comparison_returns_raw.shape)
print("Date range:", comparison_returns_raw.index.min().date(), "to", comparison_returns_raw.index.max().date())
print("Policies:", comparison_returns_raw["policy_name"].nunique())

print("\nNotebook 10 top rankings:")
print(
    comparison_rank_df[
        [
            "policy_name",
            "policy_type",
            "n_days",
            "total_return",
            "annual_return",
            "annual_volatility",
            "sharpe_ratio",
            "sortino_ratio",
            "max_drawdown",
            "calmar_ratio",
            "allocation_composite_rank",
        ]
    ].head(20).to_string(index=False)
)

# Wide return matrix.
return_wide = (
    comparison_returns_raw
    .reset_index()
    .pivot_table(index="date", columns="policy_name", values="net_return", aggfunc="last")
    .sort_index()
)

return_wide = return_wide.replace([np.inf, -np.inf], np.nan)

available_policies = return_wide.columns.tolist()

if PRIMARY_AURORA_POLICY not in available_policies:
    raise ValueError(
        f"Primary AURORA policy {PRIMARY_AURORA_POLICY} not found. "
        f"Available policies: {available_policies}"
    )

available_primary_benchmarks = [b for b in PRIMARY_BENCHMARKS if b in available_policies]
available_additional_aurora = [p for p in ADDITIONAL_AURORA_POLICIES if p in available_policies]

print("\nPrimary AURORA policy:", PRIMARY_AURORA_POLICY)
print("Available benchmarks :", available_primary_benchmarks)
print("Additional AURORA    :", available_additional_aurora)

# Restrict to common non-missing dates for all main policies.
main_policy_set = [PRIMARY_AURORA_POLICY] + available_primary_benchmarks + available_additional_aurora
main_policy_set = list(dict.fromkeys(main_policy_set))

return_wide_main = return_wide[main_policy_set].dropna(how="any").copy()

print("\nCommon return matrix:", return_wide_main.shape)
print("Common date range   :", return_wide_main.index.min().date(), "to", return_wide_main.index.max().date())

# ============================================================
# 5. Baseline observed paired statistics
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Computing observed paired statistics")
print("=" * 80)

observed_pair_rows = []

for benchmark in available_primary_benchmarks:
    obs = paired_difference_metrics(
        return_wide_main[PRIMARY_AURORA_POLICY],
        return_wide_main[benchmark],
    )

    obs.update({
        "run_id": RUN_ID,
        "aurora_policy": PRIMARY_AURORA_POLICY,
        "benchmark_policy": benchmark,
        "comparison_type": "primary_aurora_vs_primary_benchmark",
    })

    observed_pair_rows.append(obs)

for aurora_policy in available_additional_aurora:
    obs = paired_difference_metrics(
        return_wide_main[aurora_policy],
        return_wide_main[available_primary_benchmarks[0]],
    )

    obs.update({
        "run_id": RUN_ID,
        "aurora_policy": aurora_policy,
        "benchmark_policy": available_primary_benchmarks[0],
        "comparison_type": "additional_aurora_vs_best_benchmark",
    })

    observed_pair_rows.append(obs)

observed_pairs_df = pd.DataFrame(observed_pair_rows)

observed_pairs_df.to_csv(TABLE_RUN_DIR / "observed_paired_performance_differences.csv", index=False)
observed_pairs_df.to_csv(TABLE_DIR / f"table_97_observed_paired_performance_differences_{RUN_ID}.csv", index=False)

print("Observed paired differences:")
print(
    observed_pairs_df[
        [
            "aurora_policy",
            "benchmark_policy",
            "n_days",
            "diff_total_return",
            "diff_annual_return",
            "diff_sharpe",
            "diff_sortino",
            "drawdown_improvement",
            "diff_calmar",
            "annualized_mean_excess_return",
            "excess_return_hit_rate",
        ]
    ].to_string(index=False)
)

# ============================================================
# 6. HAC/Newey-West paired mean excess return tests
# ============================================================

print("\n" + "=" * 80)
print("Step 3: HAC/Newey-West paired mean excess return tests")
print("=" * 80)

hac_rows = []

for benchmark in available_primary_benchmarks:
    d = return_wide_main[PRIMARY_AURORA_POLICY] - return_wide_main[benchmark]

    hac = newey_west_mean_test(d)
    hac.update({
        "run_id": RUN_ID,
        "aurora_policy": PRIMARY_AURORA_POLICY,
        "benchmark_policy": benchmark,
        "comparison_type": "primary_aurora_vs_primary_benchmark",
    })

    if HAS_SCIPY_STATS:
        t_res = stats.ttest_1samp(d.dropna().values, popmean=0.0)
        hac["iid_t_stat"] = float(t_res.statistic)
        hac["iid_t_p_value_two_sided"] = float(t_res.pvalue)
    else:
        hac["iid_t_stat"] = np.nan
        hac["iid_t_p_value_two_sided"] = np.nan

    hac_rows.append(hac)

hac_df = pd.DataFrame(hac_rows)

hac_df.to_csv(TABLE_RUN_DIR / "hac_newey_west_mean_excess_return_tests.csv", index=False)
hac_df.to_csv(TABLE_DIR / f"table_98_hac_newey_west_mean_excess_return_tests_{RUN_ID}.csv", index=False)

print("HAC tests:")
print(
    hac_df[
        [
            "aurora_policy",
            "benchmark_policy",
            "n",
            "annualized_mean",
            "nw_se",
            "t_stat",
            "p_value_two_sided_normal",
            "iid_t_stat",
            "iid_t_p_value_two_sided",
            "max_lag",
        ]
    ].to_string(index=False)
)

# ============================================================
# 7. Block bootstrap paired tests
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Running paired circular block bootstrap")
print("=" * 80)

rng = np.random.default_rng(RANDOM_STATE)

bootstrap_metric_names = [
    "diff_total_return",
    "diff_annual_return",
    "diff_annual_volatility",
    "diff_sharpe",
    "diff_sortino",
    "drawdown_improvement",
    "diff_calmar",
    "annualized_mean_excess_return",
    "excess_return_hit_rate",
]

bootstrap_summary_rows = []
bootstrap_sample_frames = []

def run_pair_bootstrap(aurora_policy, benchmark_policy, block_length, n_bootstrap, rng):
    a = return_wide_main[aurora_policy].dropna()
    b = return_wide_main[benchmark_policy].dropna()
    common = a.index.intersection(b.index)

    a = a.loc[common].values
    b = b.loc[common].values

    n = len(common)

    rows = []

    for i in range(n_bootstrap):
        idx = circular_block_indices(n=n, block_length=block_length, rng=rng)

        a_boot = pd.Series(a[idx])
        b_boot = pd.Series(b[idx])

        metrics = paired_difference_metrics(a_boot, b_boot)

        row = {
            "bootstrap_id": i,
            "block_length": block_length,
            "aurora_policy": aurora_policy,
            "benchmark_policy": benchmark_policy,
        }

        for m in bootstrap_metric_names:
            row[m] = metrics.get(m, np.nan)

        rows.append(row)

    return pd.DataFrame(rows)

for benchmark in available_primary_benchmarks:
    for block_length in BLOCK_LENGTHS:
        print(f"Bootstrap: {PRIMARY_AURORA_POLICY} vs {benchmark}, block={block_length}")

        boot_df = run_pair_bootstrap(
            aurora_policy=PRIMARY_AURORA_POLICY,
            benchmark_policy=benchmark,
            block_length=block_length,
            n_bootstrap=N_BOOTSTRAP,
            rng=rng,
        )

        if block_length == PRIMARY_BLOCK_LENGTH:
            sample_path = BOOTSTRAP_DIR / f"bootstrap_samples_{safe_name(PRIMARY_AURORA_POLICY)}_vs_{safe_name(benchmark)}_block{block_length}.parquet"
            boot_df.to_parquet(sample_path)

            sample_csv_path = BOOTSTRAP_DIR / f"bootstrap_samples_{safe_name(PRIMARY_AURORA_POLICY)}_vs_{safe_name(benchmark)}_block{block_length}.csv"
            boot_df.to_csv(sample_csv_path, index=False)

        for metric_name in bootstrap_metric_names:
            samples = boot_df[metric_name].replace([np.inf, -np.inf], np.nan).dropna().values
            obs_row = observed_pairs_df[
                (observed_pairs_df["aurora_policy"] == PRIMARY_AURORA_POLICY)
                & (observed_pairs_df["benchmark_policy"] == benchmark)
            ].iloc[0]

            observed_value = float(obs_row[metric_name])

            ci90_lo, ci90_hi = confidence_interval(samples, 0.90)
            ci95_lo, ci95_hi = confidence_interval(samples, 0.95)
            p_one_positive, p_two = bootstrap_p_value_greater_zero(samples)

            bootstrap_summary_rows.append({
                "run_id": RUN_ID,
                "aurora_policy": PRIMARY_AURORA_POLICY,
                "benchmark_policy": benchmark,
                "block_length": block_length,
                "n_bootstrap": int(N_BOOTSTRAP),
                "metric": metric_name,
                "observed_value": observed_value,
                "bootstrap_mean": float(np.nanmean(samples)) if len(samples) else np.nan,
                "bootstrap_std": float(np.nanstd(samples, ddof=1)) if len(samples) > 1 else np.nan,
                "ci90_lower": ci90_lo,
                "ci90_upper": ci90_hi,
                "ci95_lower": ci95_lo,
                "ci95_upper": ci95_hi,
                "p_one_sided_positive": p_one_positive,
                "p_two_sided_around_zero": p_two,
                "probability_metric_positive": float(np.mean(samples > 0.0)) if len(samples) else np.nan,
            })

bootstrap_summary_df = pd.DataFrame(bootstrap_summary_rows)

bootstrap_summary_df.to_csv(TABLE_RUN_DIR / "block_bootstrap_summary_all_blocks.csv", index=False)
bootstrap_summary_df.to_csv(TABLE_DIR / f"table_99_block_bootstrap_summary_all_blocks_{RUN_ID}.csv", index=False)

primary_bootstrap_summary_df = bootstrap_summary_df[
    bootstrap_summary_df["block_length"] == PRIMARY_BLOCK_LENGTH
].copy()

primary_bootstrap_summary_df.to_csv(TABLE_RUN_DIR / "block_bootstrap_summary_primary_block.csv", index=False)
primary_bootstrap_summary_df.to_csv(TABLE_DIR / f"table_100_block_bootstrap_summary_primary_block_{RUN_ID}.csv", index=False)

print("\nPrimary block bootstrap summary:")
print(
    primary_bootstrap_summary_df[
        [
            "aurora_policy",
            "benchmark_policy",
            "metric",
            "observed_value",
            "ci95_lower",
            "ci95_upper",
            "probability_metric_positive",
            "p_one_sided_positive",
            "p_two_sided_around_zero",
        ]
    ].to_string(index=False)
)

# ============================================================
# 8. I.I.D. bootstrap sensitivity
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Running IID bootstrap sensitivity")
print("=" * 80)

iid_summary_rows = []

def run_pair_iid_bootstrap(aurora_policy, benchmark_policy, n_bootstrap, rng):
    a = return_wide_main[aurora_policy].dropna()
    b = return_wide_main[benchmark_policy].dropna()
    common = a.index.intersection(b.index)

    a = a.loc[common].values
    b = b.loc[common].values

    n = len(common)

    rows = []

    for i in range(n_bootstrap):
        idx = iid_indices(n=n, rng=rng)

        a_boot = pd.Series(a[idx])
        b_boot = pd.Series(b[idx])

        metrics = paired_difference_metrics(a_boot, b_boot)

        row = {
            "bootstrap_id": i,
            "aurora_policy": aurora_policy,
            "benchmark_policy": benchmark_policy,
        }

        for m in bootstrap_metric_names:
            row[m] = metrics.get(m, np.nan)

        rows.append(row)

    return pd.DataFrame(rows)

for benchmark in available_primary_benchmarks:
    print(f"IID bootstrap: {PRIMARY_AURORA_POLICY} vs {benchmark}")

    boot_df = run_pair_iid_bootstrap(
        aurora_policy=PRIMARY_AURORA_POLICY,
        benchmark_policy=benchmark,
        n_bootstrap=N_BOOTSTRAP,
        rng=rng,
    )

    for metric_name in bootstrap_metric_names:
        samples = boot_df[metric_name].replace([np.inf, -np.inf], np.nan).dropna().values

        obs_row = observed_pairs_df[
            (observed_pairs_df["aurora_policy"] == PRIMARY_AURORA_POLICY)
            & (observed_pairs_df["benchmark_policy"] == benchmark)
        ].iloc[0]

        observed_value = float(obs_row[metric_name])

        ci95_lo, ci95_hi = confidence_interval(samples, 0.95)
        p_one_positive, p_two = bootstrap_p_value_greater_zero(samples)

        iid_summary_rows.append({
            "run_id": RUN_ID,
            "aurora_policy": PRIMARY_AURORA_POLICY,
            "benchmark_policy": benchmark,
            "metric": metric_name,
            "observed_value": observed_value,
            "iid_bootstrap_mean": float(np.nanmean(samples)) if len(samples) else np.nan,
            "iid_bootstrap_std": float(np.nanstd(samples, ddof=1)) if len(samples) > 1 else np.nan,
            "ci95_lower": ci95_lo,
            "ci95_upper": ci95_hi,
            "probability_metric_positive": float(np.mean(samples > 0.0)) if len(samples) else np.nan,
            "p_one_sided_positive": p_one_positive,
            "p_two_sided_around_zero": p_two,
        })

iid_summary_df = pd.DataFrame(iid_summary_rows)

iid_summary_df.to_csv(TABLE_RUN_DIR / "iid_bootstrap_sensitivity_summary.csv", index=False)
iid_summary_df.to_csv(TABLE_DIR / f"table_101_iid_bootstrap_sensitivity_summary_{RUN_ID}.csv", index=False)

print("IID bootstrap summary preview:")
print(
    iid_summary_df[
        [
            "aurora_policy",
            "benchmark_policy",
            "metric",
            "observed_value",
            "ci95_lower",
            "ci95_upper",
            "probability_metric_positive",
        ]
    ].head(40).to_string(index=False)
)

# ============================================================
# 9. Statistical decision table for paper
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Creating paper decision table")
print("=" * 80)

decision_rows = []

paper_metrics = [
    "diff_total_return",
    "diff_sharpe",
    "diff_sortino",
    "drawdown_improvement",
    "diff_calmar",
    "annualized_mean_excess_return",
]

for benchmark in available_primary_benchmarks:
    for metric_name in paper_metrics:
        row = primary_bootstrap_summary_df[
            (primary_bootstrap_summary_df["benchmark_policy"] == benchmark)
            & (primary_bootstrap_summary_df["metric"] == metric_name)
        ]

        if row.empty:
            continue

        row = row.iloc[0]

        observed_value = float(row["observed_value"])
        ci95_lower = float(row["ci95_lower"])
        ci95_upper = float(row["ci95_upper"])
        prob_positive = float(row["probability_metric_positive"])
        p_one = float(row["p_one_sided_positive"])

        if metric_name in ["diff_sharpe", "diff_sortino", "drawdown_improvement", "diff_calmar"]:
            desired_direction = "positive"
        elif metric_name in ["diff_total_return", "annualized_mean_excess_return"]:
            desired_direction = "positive"
        else:
            desired_direction = "positive"

        ci_excludes_zero_positive = ci95_lower > 0.0
        ci_excludes_zero_negative = ci95_upper < 0.0

        if ci_excludes_zero_positive:
            conclusion = "significantly_positive_at_95pct_bootstrap"
        elif ci_excludes_zero_negative:
            conclusion = "significantly_negative_at_95pct_bootstrap"
        else:
            conclusion = "not_significant_at_95pct_bootstrap"

        decision_rows.append({
            "run_id": RUN_ID,
            "aurora_policy": PRIMARY_AURORA_POLICY,
            "benchmark_policy": benchmark,
            "metric": metric_name,
            "observed_value": observed_value,
            "ci95_lower": ci95_lower,
            "ci95_upper": ci95_upper,
            "probability_positive": prob_positive,
            "p_one_sided_positive": p_one,
            "conclusion": conclusion,
            "interpretation": (
                "A positive value favors AURORA for this metric. "
                "For drawdown_improvement, positive means AURORA has less severe drawdown."
            ),
        })

decision_df = pd.DataFrame(decision_rows)

decision_df.to_csv(TABLE_RUN_DIR / "paper_statistical_decision_table.csv", index=False)
decision_df.to_csv(TABLE_DIR / f"table_102_paper_statistical_decision_table_{RUN_ID}.csv", index=False)

print("Paper decision table:")
print(
    decision_df[
        [
            "benchmark_policy",
            "metric",
            "observed_value",
            "ci95_lower",
            "ci95_upper",
            "probability_positive",
            "conclusion",
        ]
    ].to_string(index=False)
)

# ============================================================
# 10. Rolling metrics and drawdown diagnostics
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Rolling and drawdown diagnostics")
print("=" * 80)

ROLLING_WINDOWS = [21, 63]

rolling_rows = []

for policy in main_policy_set:
    if policy not in return_wide_main.columns:
        continue

    r = return_wide_main[policy].dropna()

    for window in ROLLING_WINDOWS:
        rolling_mean = r.rolling(window).mean() * ANNUALIZATION_DAYS
        rolling_vol = r.rolling(window).std() * np.sqrt(ANNUALIZATION_DAYS)
        rolling_sharpe = rolling_mean / rolling_vol.replace(0.0, np.nan)

        tmp = pd.DataFrame({
            "date": r.index,
            "policy_name": policy,
            "window": window,
            "rolling_annual_return": rolling_mean.values,
            "rolling_annual_volatility": rolling_vol.values,
            "rolling_sharpe": rolling_sharpe.values,
        })

        rolling_rows.append(tmp)

rolling_df = pd.concat(rolling_rows, axis=0)
rolling_df = rolling_df.dropna(subset=["rolling_sharpe"], how="all")

rolling_df.to_csv(DIAGNOSTIC_DIR / "rolling_performance_diagnostics.csv", index=False)
rolling_df.to_csv(TABLE_DIR / f"table_103_rolling_performance_diagnostics_{RUN_ID}.csv", index=False)

drawdown_rows = []

for policy in main_policy_set:
    if policy not in return_wide_main.columns:
        continue

    r = return_wide_main[policy].dropna()
    eq = equity_curve(r)
    dd = eq / eq.cummax() - 1.0

    drawdown_rows.append(pd.DataFrame({
        "date": dd.index,
        "policy_name": policy,
        "equity": eq.values,
        "drawdown": dd.values,
    }))

drawdown_df = pd.concat(drawdown_rows, axis=0)

drawdown_df.to_csv(DIAGNOSTIC_DIR / "equity_and_drawdown_diagnostics.csv", index=False)
drawdown_df.to_csv(TABLE_DIR / f"table_104_equity_and_drawdown_diagnostics_{RUN_ID}.csv", index=False)

# ============================================================
# 11. Plots and paper figures
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Creating plots and paper figures")
print("=" * 80)

def plot_bootstrap_distribution(benchmark, metric_name, path):
    sample_path = BOOTSTRAP_DIR / f"bootstrap_samples_{safe_name(PRIMARY_AURORA_POLICY)}_vs_{safe_name(benchmark)}_block{PRIMARY_BLOCK_LENGTH}.parquet"

    if not sample_path.exists():
        return

    boot_df = pd.read_parquet(sample_path)

    if metric_name not in boot_df.columns:
        return

    samples = boot_df[metric_name].replace([np.inf, -np.inf], np.nan).dropna()

    if len(samples) == 0:
        return

    obs_row = observed_pairs_df[
        (observed_pairs_df["aurora_policy"] == PRIMARY_AURORA_POLICY)
        & (observed_pairs_df["benchmark_policy"] == benchmark)
    ].iloc[0]

    observed = obs_row[metric_name]

    ci95_lo, ci95_hi = confidence_interval(samples.values, 0.95)

    plt.figure(figsize=(9, 5))
    sns.histplot(samples, bins=50, kde=True, color="#4C72B0")
    plt.axvline(0.0, color="black", linestyle="--", linewidth=1.2, label="zero")
    plt.axvline(observed, color="red", linestyle="-", linewidth=1.8, label="observed")
    plt.axvline(ci95_lo, color="gray", linestyle=":", linewidth=1.5, label="95% CI")
    plt.axvline(ci95_hi, color="gray", linestyle=":", linewidth=1.5)
    plt.title(f"Block bootstrap distribution: {metric_name}\n{PRIMARY_AURORA_POLICY} vs {benchmark}")
    plt.xlabel(metric_name)
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_equity_curves(path):
    policies = [PRIMARY_AURORA_POLICY] + available_primary_benchmarks

    plt.figure(figsize=(12, 6))

    for policy in policies:
        r = return_wide_main[policy].dropna()
        eq = equity_curve(r)
        plt.plot(eq.index, eq.values, label=policy, linewidth=1.7)

    plt.title("Aligned strict-test equity curves")
    plt.xlabel("Date")
    plt.ylabel("Equity, initial capital = 1")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_drawdowns(path):
    policies = [PRIMARY_AURORA_POLICY] + available_primary_benchmarks

    plt.figure(figsize=(12, 6))

    for policy in policies:
        r = return_wide_main[policy].dropna()
        eq = equity_curve(r)
        dd = eq / eq.cummax() - 1.0
        plt.plot(dd.index, dd.values, label=policy, linewidth=1.5)

    plt.title("Aligned strict-test drawdowns")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_rolling_sharpe(window, path):
    tmp = rolling_df[
        (rolling_df["window"] == window)
        & (rolling_df["policy_name"].isin([PRIMARY_AURORA_POLICY] + available_primary_benchmarks))
    ].copy()

    plt.figure(figsize=(12, 6))

    for policy, grp in tmp.groupby("policy_name"):
        grp = grp.sort_values("date")
        plt.plot(grp["date"], grp["rolling_sharpe"], label=policy, linewidth=1.5)

    plt.title(f"Rolling {window}-day annualized Sharpe")
    plt.xlabel("Date")
    plt.ylabel("Rolling Sharpe")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_decision_heatmap(path):
    tmp = decision_df.copy()

    metric_order = [
        "diff_total_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
        "diff_calmar",
        "annualized_mean_excess_return",
    ]

    pivot = tmp.pivot_table(
        index="metric",
        columns="benchmark_policy",
        values="observed_value",
        aggfunc="first",
    ).reindex(metric_order)

    plt.figure(figsize=(11, 5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        center=0.0,
        linewidths=0.5,
    )
    plt.title("Observed AURORA minus benchmark differences")
    plt.xlabel("Benchmark")
    plt.ylabel("Metric")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

plot_equity_curves(PLOT_DIR / "notebook11_equity_curves_primary_comparison.png")
plot_drawdowns(PLOT_DIR / "notebook11_drawdowns_primary_comparison.png")
plot_rolling_sharpe(21, PLOT_DIR / "notebook11_rolling_21d_sharpe.png")
plot_rolling_sharpe(63, PLOT_DIR / "notebook11_rolling_63d_sharpe.png")
plot_decision_heatmap(PLOT_DIR / "notebook11_observed_difference_heatmap.png")

for benchmark in available_primary_benchmarks:
    for metric_name in ["diff_sharpe", "diff_sortino", "drawdown_improvement", "diff_total_return"]:
        plot_bootstrap_distribution(
            benchmark=benchmark,
            metric_name=metric_name,
            path=PLOT_DIR / f"bootstrap_distribution_{safe_name(metric_name)}_{safe_name(benchmark)}.png",
        )

paper_figure_files = [
    "notebook11_equity_curves_primary_comparison.png",
    "notebook11_drawdowns_primary_comparison.png",
    "notebook11_rolling_21d_sharpe.png",
    "notebook11_rolling_63d_sharpe.png",
    "notebook11_observed_difference_heatmap.png",
]

for fname in paper_figure_files:
    src = PLOT_DIR / fname
    if src.exists():
        dst = PAPER_FIGURE_DIR / fname
        dst.write_bytes(src.read_bytes())

        global_dst = FIGURE_DIR / f"{Path(fname).stem}_{RUN_ID}.png"
        global_dst.write_bytes(src.read_bytes())

for src in PLOT_DIR.glob("bootstrap_distribution_*.png"):
    dst = PAPER_FIGURE_DIR / src.name
    dst.write_bytes(src.read_bytes())

    global_dst = FIGURE_DIR / f"{src.stem}_{RUN_ID}.png"
    global_dst.write_bytes(src.read_bytes())

print("Plots saved to        :", PLOT_DIR)
print("Paper figures saved to:", PAPER_FIGURE_DIR)

# ============================================================
# 12. Final diagnostic summary
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Creating final diagnostic summary")
print("=" * 80)

primary_best_benchmark = available_primary_benchmarks[0]

primary_decisions = decision_df[
    decision_df["benchmark_policy"] == primary_best_benchmark
].copy()

def get_decision(metric):
    row = primary_decisions[primary_decisions["metric"] == metric]
    if row.empty:
        return "not_available"
    return row.iloc[0]["conclusion"]

diag_rows = [
    {
        "question": "What is the primary AURORA policy tested?",
        "finding": PRIMARY_AURORA_POLICY,
        "evidence": "This was the best UAMV policy in Notebook 10 by aligned strict-test composite rank.",
    },
    {
        "question": "What is the primary benchmark?",
        "finding": primary_best_benchmark,
        "evidence": "This benchmark was the strongest benchmark comparator from the Notebook 10 ranking table.",
    },
    {
        "question": "Is AURORA total-return superiority statistically supported?",
        "finding": get_decision("diff_total_return"),
        "evidence": "Based on paired circular block bootstrap with the primary block length.",
    },
    {
        "question": "Is AURORA Sharpe improvement statistically supported?",
        "finding": get_decision("diff_sharpe"),
        "evidence": "Based on paired circular block bootstrap with the primary block length.",
    },
    {
        "question": "Is AURORA Sortino improvement statistically supported?",
        "finding": get_decision("diff_sortino"),
        "evidence": "Based on paired circular block bootstrap with the primary block length.",
    },
    {
        "question": "Is AURORA drawdown improvement statistically supported?",
        "finding": get_decision("drawdown_improvement"),
        "evidence": "Positive drawdown improvement means AURORA has less severe maximum drawdown.",
    },
    {
        "question": "What is the methodological caveat?",
        "finding": "Short test period and multiple strategy comparisons",
        "evidence": (
            "The aligned strict-test period contains 319 trading days. "
            "Bootstrap results should be interpreted as empirical uncertainty diagnostics, "
            "not as proof of future outperformance."
        ),
    },
]

diagnostic_df = pd.DataFrame(diag_rows)

diagnostic_df.to_csv(TABLE_RUN_DIR / "notebook11_diagnostic_summary.csv", index=False)
diagnostic_df.to_csv(TABLE_DIR / f"table_105_notebook11_diagnostic_summary_{RUN_ID}.csv", index=False)

print("Diagnostic summary:")
print(diagnostic_df.to_string(index=False))

# ============================================================
# 13. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 10: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "11_AURORA_statistical_significance_and_block_bootstrap.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook10_run_id": NOTEBOOK10_RUN_ID,
    "notebook10_root": str(NOTEBOOK10_ROOT),
    "comparison_returns_path": str(NOTEBOOK10_COMPARISON_RETURNS),
    "comparison_rankings_path": str(NOTEBOOK10_COMPARISON_RANKINGS),
    "primary_aurora_policy": PRIMARY_AURORA_POLICY,
    "primary_benchmarks": available_primary_benchmarks,
    "additional_aurora_policies": available_additional_aurora,
    "n_bootstrap": N_BOOTSTRAP,
    "block_lengths": BLOCK_LENGTHS,
    "primary_block_length": PRIMARY_BLOCK_LENGTH,
    "random_state": RANDOM_STATE,
    "confidence_levels": CONFIDENCE_LEVELS,
    "n_common_days": int(return_wide_main.shape[0]),
    "common_start_date": str(return_wide_main.index.min().date()),
    "common_end_date": str(return_wide_main.index.max().date()),
    "important_methodological_note": (
        "Paired circular block bootstrap is used to account for serial dependence in daily returns. "
        "Results remain exploratory because the strict-test sample is short and strategy selection "
        "was partially informed by prior experimental results."
    ),
    "diagnostics": diagnostic_df.to_dict(orient="records"),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "reports": str(REPORT_RUN_DIR),
        "plots": str(PLOT_DIR),
        "paper_figures": str(PAPER_FIGURE_DIR),
        "bootstrap_samples": str(BOOTSTRAP_DIR),
    },
    "educational_note": (
        "This notebook performs research-oriented statistical diagnostics only and does not provide personalized financial advice."
    ),
}

validation_report_path = REPORT_RUN_DIR / "AURORA_11_statistical_significance_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_11_statistical_significance_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_11_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_11_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 14. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 11 COMPLETE")
print("=" * 80)
print("Run ID                                  :", RUN_ID)
print("Run root                                :", RUN_ROOT)
print("Observed paired differences             :", TABLE_DIR / f"table_97_observed_paired_performance_differences_{RUN_ID}.csv")
print("HAC/Newey-West tests                    :", TABLE_DIR / f"table_98_hac_newey_west_mean_excess_return_tests_{RUN_ID}.csv")
print("Block bootstrap all blocks              :", TABLE_DIR / f"table_99_block_bootstrap_summary_all_blocks_{RUN_ID}.csv")
print("Block bootstrap primary block           :", TABLE_DIR / f"table_100_block_bootstrap_summary_primary_block_{RUN_ID}.csv")
print("IID bootstrap sensitivity               :", TABLE_DIR / f"table_101_iid_bootstrap_sensitivity_summary_{RUN_ID}.csv")
print("Paper statistical decision table        :", TABLE_DIR / f"table_102_paper_statistical_decision_table_{RUN_ID}.csv")
print("Rolling performance diagnostics         :", TABLE_DIR / f"table_103_rolling_performance_diagnostics_{RUN_ID}.csv")
print("Equity and drawdown diagnostics         :", TABLE_DIR / f"table_104_equity_and_drawdown_diagnostics_{RUN_ID}.csv")
print("Notebook 11 diagnostic summary          :", TABLE_DIR / f"table_105_notebook11_diagnostic_summary_{RUN_ID}.csv")
print("Plots directory                         :", PLOT_DIR)
print("Paper figures directory                 :", PAPER_FIGURE_DIR)
print("Bootstrap samples directory             :", BOOTSTRAP_DIR)
print("Validation report                       :", validation_report_path)
print("Manifest                                :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("12_AURORA_paper_tables_figures_and_manuscript_assets.ipynb")

Mounted at /content/drive
AURORA-TWETF Notebook 11: Statistical Significance and Block Bootstrap
Timestamp UTC              : 2026-06-24T12:48:34Z
Run ID                     : 20260624_124834
Notebook 10 root           : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748
Notebook 10 comparison ret : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748/returns/comparison_returns_uamv_vs_benchmarks.parquet
Notebook 10 rankings       : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/tables/table_93_comparison_rankings_uamv_vs_benchmarks_20260624_100748.csv
Run root                   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/statistical_significance_block_bootstrap/run_20260624_124834

Step 1: Loading Notebook 10 comparison returns and rankings
Comparison returns shape: (7975, 12)
Date range: 2024-11-27 to 2026-03-25
Policies: 25
